In [0]:
%sql
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1'

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:5]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(1)
158


Enable liquid clustering

In [0]:
%sql
alter table nyc_taxi cluster by (trip_distance)

In [0]:
%sql
describe history nyc_taxi

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-31T15:11:13Z,147836707444603,anooptu@gmail.com,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> trip_distance)",null,List(3322194627801978),0826-063441-ydade36q,3,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-31T15:11:12Z,147836707444603,anooptu@gmail.com,ROW TRACKING BACKFILL,Map(batchId -> 0),null,List(3322194627801978),0826-063441-ydade36q,2,SnapshotIsolation,false,Map(),null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-31T15:11:09Z,147836707444603,anooptu@gmail.com,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})",null,List(3322194627801978),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-31T15:10:14Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3322194627801978),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numFiles -> 200, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 45142478, numOutputBytes -> 1604461403)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-31T15:05:37Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3322194627801978),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


Liquid clustering is just enabled, and not executed, so we still have 200 files

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,8000010.5,part-00032-68779062-442c-4250-888e-aaade0244c5a-c000.zstd.parquet
0.0,64.8,part-00015-47dfcc63-2b9c-48c4-9925-6c1ba97caa43-c000.zstd.parquet
0.0,183.1,part-00093-916b123d-8382-4f35-90a1-8b87b40197d7-c000.zstd.parquet
0.0,173.9,part-00184-b3bcc980-17d4-4c57-aef5-564d36a3af00-c000.zstd.parquet
0.0,56.84,part-00029-c7de332a-52ce-4e06-bd32-decbbf392d90-c000.zstd.parquet
0.0,87.0,part-00154-437dd23a-5256-4efd-a8c7-4f972e9d6834-c000.zstd.parquet
0.0,54.49,part-00060-e43c012a-a82f-47bb-9393-38794351b90f-c000.zstd.parquet
0.0,84.2,part-00030-b8bf2217-8411-4f7b-a91d-2e90d7696780-c000.zstd.parquet
0.0,107.6,part-00159-4b7e772b-4ddf-4bdb-ad8d-eef0eac4f80a-c000.zstd.parquet
0.0,202.0,part-00024-9dc806ee-ed70-4cc9-9d96-4b2735d2841d-c000.zstd.parquet


OPTIMIZE will actually execute the liquid clustering after enabling it

In [0]:
%sql
optimize nyc_taxi

path,metrics
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1,"List(7, 200, List(186192313, 467425464, 2.5621797657142857E8, 7, 1793525836), List(7984771, 8078634, 8022307.015, 200, 1604461403), 0, null, null, 0, 1, 200, 0, false, 0, 0, 1788189256407, 1788189519069, 4, 1, null, List(0, 0), null, 18, 18, 472556, 0, List(1604461403, true, false, false, 0.8178132950211963, List(0.8178132950211963), 1.0, 0, 200, 1604461403, 1604461403, 0, 0, 0, null, log, 16777216, 268435456, 8, 0, 0, List(0), 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1604461403, 1604461403, List(322, 17392, 0, 561, 16650, 30013), 2, 1, 5, sizeAware))"
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 7, 0, false, 0, 0, 1788189519118, 1788189523259, 4, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1793525836, false, false, false, 0.8178132950211963, List(0.8178132950211963), 1.0, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 268435456, 8, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(245, 32, 632, 0, 0, 0), 2, 2, 5, sizeAware))"


In [0]:
%sql
describe detail nyc_taxi;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c0b713da-7843-41b8-8abe-abed81e2b444,useastws.default.nyc_taxi,null,abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1,2026-08-31T15:05:36.961Z,2026-08-31T15:18:38Z,List(),List(trip_distance),7,1793525836,"Map(delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-b4406335-4f7e-4e2b-aa18-3c6d316c5d43, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-4b3e6179-146b-47cd-91dc-ff51e76e99ba)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe history nyc_taxi

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-08-31T15:18:38Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""trip_distance""], isFull -> false, zOrderBy -> [], batchId -> 0)",null,List(3322194627801978),0826-063441-ydade36q,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 200, numRemovedBytes -> 1604461403, p25FileSize -> 195597535, numDeletionVectorsRemoved -> 0, conflictDetectionTimeMs -> 61, minFileSize -> 186192313, numAddedFiles -> 7, maxFileSize -> 467425464, p75FileSize -> 296389354, p50FileSize -> 204865098, numAddedBytes -> 1793525836)",null,Databricks-Runtime/16.4.x-scala2.13
5,2026-08-31T15:14:48Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""trip_distance""], isFull -> false, zOrderBy -> [], batchId -> -1)",null,List(3322194627801978),0826-063441-ydade36q,4,SnapshotIsolation,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
4,2026-08-31T15:11:13Z,147836707444603,anooptu@gmail.com,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> trip_distance)",null,List(3322194627801978),0826-063441-ydade36q,3,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-31T15:11:12Z,147836707444603,anooptu@gmail.com,ROW TRACKING BACKFILL,Map(batchId -> 0),null,List(3322194627801978),0826-063441-ydade36q,2,SnapshotIsolation,false,Map(),null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-31T15:11:09Z,147836707444603,anooptu@gmail.com,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})",null,List(3322194627801978),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-31T15:10:14Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3322194627801978),0826-063441-ydade36q,0,WriteSerializable,false,"Map(numFiles -> 200, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 45142478, numOutputBytes -> 1604461403)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-31T15:05:37Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3322194627801978),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.8999999999999999,part-00001-decdc3f8-adbb-495b-a881-47c757ad68dc-c000.snappy.parquet
0.9,1.228,part-00002-41937cfb-a93a-4354-9013-eebf6b6d95fc-c000.snappy.parquet
1.23,1.6,part-00005-ada8bb12-e859-45a6-a4aa-c91e653b3335-c000.snappy.parquet
1.61,2.14,part-00004-3c1ac31f-044f-493b-8811-b38108ab3cdd-c000.snappy.parquet
2.15,3.09,part-00003-48b9e102-f761-43f7-b177-5074bf730693-c000.snappy.parquet
3.1,3.95,part-00006-4adbd4db-66a5-4529-9fbd-95931c68d5e7-c000.snappy.parquet
3.96,1.18000006E7,part-00000-892c6d38-7639-4921-b690-94752987e206-c000.snappy.parquet


In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(1)
158


Query ran much faster as we have now liquid clustering activated on the table. Just 7 files and has ordering of range

In [0]:
%sql
alter table nyc_taxi cluster by auto;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5870122210059479>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'alter table nyc_taxi cluster by auto;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     raise Exception(
    127         "Cannot run %sql command becau

Remove clustering

In [0]:
%sql
alter table nyc_taxi cluster by none;